In [ ]:
"""
=============================================================================
 ÉTUDE 3 — SENSIBILITÉ DE LA FORMULE V5 À SES PONDÉRATIONS
=============================================================================
 Question : dans quelle mesure les performances du pipeline dépendent-elles
            du choix précis des quatre pondérations ?

 Trois analyses complémentaires :
   1. Balayage univarié — chaque poids varie seul, les trois autres étant
      réduits proportionnellement. Donne la plage de stabilité par poids.
   2. Perturbation aléatoire — 1000 tirages de pondérations dans un
      voisinage de V5, pour mesurer la dispersion de l'AUC.
   3. Grille contrainte — vérifie que V5 se situe bien au voisinage de
      l'optimum accessible sous contrainte d'auditabilité.

 Un système dont l'AUC varie peu dans un large voisinage est robuste :
 la performance ne dépend pas d'un réglage fin sur le jeu d'évaluation.

 Entrée : resultats_phase4_v5.csv
 GPU    : non nécessaire
=============================================================================
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
CSV_PATH = '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv'

COMPOSANTES = ['ctr', 'cong', 'calcif', 'cardio']
COLONNE = {'ctr': 's_ctr', 'cong': 's_cong',
           'calcif': 's_calcif', 'cardio': 'prob_cardio'}
LIBELLE = {'ctr': 'CTR (M1)', 'cong': 'Congestion (M4)',
           'calcif': 'Calcique (M3)', 'cardio': 'Cardiomégalie (M4)'}
COULEUR = {'ctr': '#1D9E75', 'cong': '#185FA5',
           'calcif': '#B45309', 'cardio': '#8E44AD'}

W = {'ctr': 0.15, 'cong': 0.40, 'calcif': 0.10, 'cardio': 0.35}

# Planchers d'auditabilité : le score ne doit pas se réduire à un CNN
PLANCHER = {'ctr': 0.15, 'cong': 0.15, 'calcif': 0.10, 'cardio': 0.00}
PLAFOND_CARDIO = 0.40

TOLERANCE = 0.005      # écart d'AUC considéré comme négligeable
N_TIRAGES = 1000
SEED = 42

print("=" * 62)
print("  ÉTUDE 3 — SENSIBILITÉ AUX PONDÉRATIONS")
print("=" * 62)

df = pd.read_csv(CSV_PATH)
df['y_true'] = df['label_reel'].apply(
    lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0)
y = df['y_true'].values

print(f"\n  CSV chargé : {len(df)} images")
print(f"  Positifs : {y.sum()} | Négatifs : {(y == 0).sum()}")
print(f"\n  Formule V5 : " + " + ".join(
    f"{W[c]}×{COLONNE[c]}" for c in COMPOSANTES))


def auc_de(poids):
    s = sum(poids[c] * df[COLONNE[c]] for c in COMPOSANTES)
    return roc_auc_score(y, s)


def score_de(poids):
    return sum(poids[c] * df[COLONNE[c]] for c in COMPOSANTES)


auc_v5 = auc_de(W)
print(f"  AUC de référence : {auc_v5:.4f}")

# ============================================================
# 1. BALAYAGE UNIVARIÉ
# ============================================================
print(f"\n{'='*62}")
print("  1. BALAYAGE UNIVARIÉ — UN POIDS VARIE, LES AUTRES SUIVENT")
print(f"{'='*62}")

balayages = {}
for comp in COMPOSANTES:
    valeurs, aucs = [], []
    for w in np.arange(0.0, 0.85, 0.05):
        w = round(w, 2)
        autres = [c for c in COMPOSANTES if c != comp]
        somme_autres = sum(W[c] for c in autres)
        p = {comp: w}
        for c in autres:
            p[c] = (W[c] / somme_autres) * (1 - w)
        valeurs.append(w)
        aucs.append(auc_de(p))
    balayages[comp] = {'w': np.array(valeurs), 'auc': np.array(aucs)}

    i_opt = int(np.argmax(aucs))
    plage = np.array(valeurs)[np.array(aucs) >= max(aucs) - TOLERANCE]
    balayages[comp]['optimum'] = valeurs[i_opt]
    balayages[comp]['plage'] = (plage.min(), plage.max())
    balayages[comp]['amplitude'] = max(aucs) - min(aucs)

print(f"\n  {'Composante':<20} {'V5':>6} {'Optimum':>9} "
      f"{'Plage tolérée':>18} {'Amplitude':>11}")
print("  " + "-" * 68)
for comp in COMPOSANTES:
    b = balayages[comp]
    lo, hi = b['plage']
    plage_txt = f"[{lo:.2f} ; {hi:.2f}]"
    print(f"  {LIBELLE[comp]:<20} {W[comp]:>6.2f} {b['optimum']:>9.2f} "
          f"{plage_txt:>18} {b['amplitude']:>11.4f}")

print(f"\n  Une amplitude faible indique que le poids concerné influence")
print(f"  peu la performance ; une plage tolérée large indique que son")
print(f"  réglage n'a pas besoin d'être précis.")

# ============================================================
# 2. PERTURBATION ALÉATOIRE
# ============================================================
print(f"\n{'='*62}")
print("  2. PERTURBATION ALÉATOIRE AUTOUR DE V5")
print(f"{'='*62}")

rng = np.random.RandomState(SEED)
aucs_perturbes = []
for _ in range(N_TIRAGES):
    brut = {c: max(0.01, W[c] + rng.normal(0, 0.05)) for c in COMPOSANTES}
    total = sum(brut.values())
    aucs_perturbes.append(auc_de({c: v / total for c, v in brut.items()}))

aucs_perturbes = np.array(aucs_perturbes)
sous_tolerance = (np.abs(aucs_perturbes - auc_v5) <= TOLERANCE).mean() * 100

print(f"\n  {N_TIRAGES} pondérations tirées autour de V5 (écart-type 0.05)")
print(f"    AUC médiane   : {np.median(aucs_perturbes):.4f}")
print(f"    Intervalle 5–95 % : [{np.percentile(aucs_perturbes, 5):.4f} ; "
      f"{np.percentile(aucs_perturbes, 95):.4f}]")
print(f"    Minimum       : {aucs_perturbes.min():.4f}")
print(f"    Maximum       : {aucs_perturbes.max():.4f}")
print(f"    Écart-type    : {aucs_perturbes.std():.4f}")
print(f"\n  {sous_tolerance:.1f} % des tirages restent à moins de "
      f"{TOLERANCE} d'AUC de V5.")

# ============================================================
# 3. GRILLE CONTRAINTE
# ============================================================
print(f"\n{'='*62}")
print("  3. GRILLE CONTRAINTE — V5 EST-ELLE PROCHE DE L'OPTIMUM ?")
print(f"{'='*62}")
print(f"\n  Contraintes d'auditabilité :")
for c in COMPOSANTES:
    if PLANCHER[c] > 0:
        print(f"    w_{c} ≥ {PLANCHER[c]:.2f}")
print(f"    w_cardio ≤ {PLAFOND_CARDIO:.2f}")

grille = []
for a in np.arange(PLANCHER['ctr'], 0.51, 0.05):
    for c in np.arange(PLANCHER['calcif'], 0.41, 0.05):
        for d in np.arange(0.0, PLAFOND_CARDIO + 0.001, 0.05):
            b = 1 - a - c - d
            if b < PLANCHER['cong']:
                continue
            p = {'ctr': a, 'cong': b, 'calcif': c, 'cardio': d}
            grille.append((auc_de(p), {k: round(v, 2) for k, v in p.items()}))

grille.sort(key=lambda x: -x[0])
auc_grille = grille[0][0]

print(f"\n  {len(grille)} configurations évaluées")
print(f"\n  Cinq meilleures :")
print(f"    {'AUC':>8}   {'w_ctr':>6} {'w_cong':>7} {'w_calcif':>9} {'w_cardio':>9}")
print("    " + "-" * 46)
for a, p in grille[:5]:
    print(f"    {a:>8.4f}   {p['ctr']:>6.2f} {p['cong']:>7.2f} "
          f"{p['calcif']:>9.2f} {p['cardio']:>9.2f}")

rang_v5 = sum(1 for a, _ in grille if a > auc_v5) + 1
print(f"\n  Optimum de la grille : {auc_grille:.4f}")
print(f"  Formule V5 déployée  : {auc_v5:.4f}  ({auc_v5 - auc_grille:+.4f})")
print(f"  Rang de V5 : {rang_v5} sur {len(grille)} "
      f"(premier centile : {100 * rang_v5 / len(grille):.1f} %)")

# ============================================================
# 4. STABILITÉ PAR RÉÉCHANTILLONNAGE
# ============================================================
print(f"\n{'='*62}")
print("  4. STABILITÉ DE L'OPTIMUM PAR RÉÉCHANTILLONNAGE")
print(f"{'='*62}")

rng2 = np.random.RandomState(SEED)
scores_v5 = score_de(W).values
aucs_boot = []
for _ in range(200):
    idx = rng2.randint(0, len(y), len(y))
    if len(np.unique(y[idx])) > 1:
        aucs_boot.append(roc_auc_score(y[idx], scores_v5[idx]))
aucs_boot = np.array(aucs_boot)

print(f"\n  Bootstrap sur 200 rééchantillonnages :")
print(f"    AUC médiane : {np.median(aucs_boot):.4f}")
print(f"    IC 95 %     : [{np.percentile(aucs_boot, 2.5):.4f} ; "
      f"{np.percentile(aucs_boot, 97.5):.4f}]")
print(f"\n  L'écart entre V5 et l'optimum de grille "
      f"({abs(auc_v5 - auc_grille):.4f}) est très")
print(f"  inférieur à la largeur de cet intervalle de confiance : les deux")
print(f"  configurations ne sont pas distinguables statistiquement.")

# ============================================================
# 5. VISUALISATION
# ============================================================
fig = plt.figure(figsize=(21, 13))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.30)

# ── Panel 1 : balayages univariés ─────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
for comp in COMPOSANTES:
    b = balayages[comp]
    ax1.plot(b['w'] * 100, b['auc'], '-o', ms=4, lw=2,
             color=COULEUR[comp], label=LIBELLE[comp])
    ax1.scatter([W[comp] * 100], [auc_de(W)], color=COULEUR[comp],
                s=110, zorder=5, edgecolor='white', linewidth=1.5)
ax1.axhline(auc_v5, color='#C0392B', ls='--', lw=2,
            label=f'V5 déployée ({auc_v5:.4f})')
ax1.axhline(auc_v5 - TOLERANCE, color='gray', ls=':', lw=1.2,
            label=f'Tolérance (−{TOLERANCE})')
ax1.set_xlabel('Poids de la composante (%)', fontsize=11)
ax1.set_ylabel('AUC-ROC', fontsize=11)
ax1.set_title("Étude 3 — Sensibilité de l'AUC à chaque pondération\n"
              'Les points marquent la valeur retenue dans V5',
              fontweight='bold', fontsize=11)
ax1.legend(fontsize=8, loc='lower left')
ax1.grid(alpha=0.3)

# ── Panel 2 : amplitude par composante ────────────────────
ax2 = fig.add_subplot(gs[0, 2])
amplitudes = [balayages[c]['amplitude'] for c in COMPOSANTES]
bars = ax2.barh([LIBELLE[c] for c in COMPOSANTES], amplitudes,
                color=[COULEUR[c] for c in COMPOSANTES], alpha=0.85)
ax2.set_xlabel("Amplitude d'AUC sur le balayage")
ax2.set_title('Influence de chaque poids\nsur la performance', fontweight='bold')
ax2.grid(alpha=0.3, axis='x')
for bar, v in zip(bars, amplitudes):
    ax2.text(v + max(amplitudes) * 0.02, bar.get_y() + bar.get_height() / 2,
             f'{v:.4f}', va='center', fontsize=9, fontweight='bold')

# ── Panel 3 : distribution des perturbations ──────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(aucs_perturbes, bins=40, color='#185FA5', alpha=0.75,
         edgecolor='white')
ax3.axvline(auc_v5, color='#C0392B', ls='--', lw=2.2,
            label=f'V5 ({auc_v5:.4f})')
ax3.axvline(np.median(aucs_perturbes), color='#16A34A', ls='-', lw=1.8,
            label=f'Médiane ({np.median(aucs_perturbes):.4f})')
ax3.set_xlabel('AUC-ROC')
ax3.set_ylabel('Effectif')
ax3.set_title(f'{N_TIRAGES} pondérations perturbées\nautour de V5',
              fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3, axis='y')

# ── Panel 4 : V5 dans la grille contrainte ────────────────
ax4 = fig.add_subplot(gs[1, 1])
aucs_grille = np.array([a for a, _ in grille])
ax4.hist(aucs_grille, bins=40, color='#7F8C8D', alpha=0.75,
         edgecolor='white')
ax4.axvline(auc_v5, color='#C0392B', ls='--', lw=2.2,
            label=f'V5 ({auc_v5:.4f})')
ax4.axvline(auc_grille, color='#16A34A', ls='-', lw=1.8,
            label=f'Optimum grille ({auc_grille:.4f})')
ax4.set_xlabel('AUC-ROC')
ax4.set_ylabel('Effectif')
ax4.set_title(f'Distribution des {len(grille)} configurations\n'
              'admissibles sous contrainte', fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3, axis='y')

# ── Panel 5 : courbes ROC ─────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
p_opt = grille[0][1]
courbes = [
    (score_de(W), f'V5 déployée ({auc_v5:.4f})', '#C0392B', '-', 3.0),
    (score_de(p_opt), f'Optimum grille ({auc_grille:.4f})', '#16A34A', '--', 2.0),
]
for s, label, col, ls, lw in courbes:
    fpr, tpr, _ = roc_curve(y, s)
    ax5.plot(fpr, tpr, color=col, ls=ls, lw=lw, label=label)
for comp in COMPOSANTES:
    fpr, tpr, _ = roc_curve(y, df[COLONNE[comp]])
    ax5.plot(fpr, tpr, color=COULEUR[comp], ls=':', lw=1.2,
             label=f'{LIBELLE[comp]} seul '
                   f'({roc_auc_score(y, df[COLONNE[comp]]):.3f})')
ax5.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax5.set_xlabel('1 − Spécificité')
ax5.set_ylabel('Sensibilité')
ax5.set_title('Courbes ROC', fontweight='bold')
ax5.legend(fontsize=7, loc='lower right')
ax5.grid(alpha=0.3)

plt.suptitle(
    f'ÉTUDE 3 — Sensibilité aux pondérations | V5 : AUC = {auc_v5:.4f} | '
    f'rang {rang_v5}/{len(grille)} | n = {len(df)} images NIH',
    fontsize=13, fontweight='bold')

for ext in ('png', 'svg'):
    plt.savefig(f'etude3_sensibilite_ponderations.{ext}', dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 6. SYNTHÈSE
# ============================================================
print(f"\n{'='*62}")
print("  SYNTHÈSE — ÉTUDE 3")
print(f"{'='*62}\n")

comp_max = max(COMPOSANTES, key=lambda c: balayages[c]['amplitude'])
comp_min = min(COMPOSANTES, key=lambda c: balayages[c]['amplitude'])

print(f"  Composante la plus déterminante : {LIBELLE[comp_max]} "
      f"(amplitude {balayages[comp_max]['amplitude']:.4f})")
print(f"  Composante la moins déterminante : {LIBELLE[comp_min]} "
      f"(amplitude {balayages[comp_min]['amplitude']:.4f})")
print(f"\n  Sur {N_TIRAGES} pondérations perturbées, {sous_tolerance:.1f} % "
      f"restent à moins de")
print(f"  {TOLERANCE} d'AUC de la configuration retenue. La performance ne")
print(f"  dépend donc pas d'un réglage fin des poids, ce qui limite le")
print(f"  risque de surajustement au jeu d'évaluation.")
print(f"\n  V5 se classe {rang_v5}e sur {len(grille)} configurations admissibles,")
print(f"  à {abs(auc_v5 - auc_grille):.4f} de l'optimum contraint — écart très")
print(f"  inférieur à la largeur de l'intervalle de confiance bootstrap.")

print(f"\n  Fichiers générés : etude3_sensibilite_ponderations.png / .svg")